In [ ]:
import gc
import io
import os
import random
import re
import urllib.request
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore")

RUNTIME_ERROR = ""
try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
except Exception as exc:
    torch = None
    AutoModelForCausalLM = None
    AutoTokenizer = None
    RUNTIME_ERROR = f"{type(exc).__name__}: {exc}"

HF_TOKEN = ""

REQUIRED_PACKAGES = ("numpy", "pandas", "torch", "transformers", "accelerate", "sentencepiece", "psutil") + (("pywin32",) if os.name == "nt" else ())

# Please provide local paths or Hugging Face identifiers for Phi-3.5-mini-instruct, Llama-3.2-1B-Instruct, DeepSeek-Coder-1.3B-Instruct, and Gemma-2B-it.
MODEL_LOCATIONS = {
    "Phi-3.5-mini": "",
    "LLaMA-3.2-1B": "",
    "DeepSeek-1.3B": "",
    "Gemma-2B-it": "",
}

MODEL_REVISIONS = {
    "Phi-3.5-mini": "",
    "LLaMA-3.2-1B": "",
    "DeepSeek-1.3B": "",
    "Gemma-2B-it": "",
}

JENA_URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip"

# For reproducibility, please prepare each non-Jena dataset to match DATA_TARGETS and EXPECTED_ROWS, then set its local CSV path below.
# JenaClimate: https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip (leave its path blank to download it and resample T (degC) to hourly means automatically)
# AzureLLMTrace: https://github.com/Azure/AzurePublicDataset/releases/download/dataset-llm-2024/AzureLLMInferenceTrace_code_1week.csv (please retain the final 200000 rows in the released file order and save their ContextTokens values as a one-column CSV)
# ETTh1: https://raw.githubusercontent.com/zhouhaoyi/ETDataset/1d16c8f4f943005d613b5bc962e9eeb06058cf07/ETT-small/ETTh1.csv (please retain the 17420 chronologically ordered OT observations)
# ETTm1: https://raw.githubusercontent.com/zhouhaoyi/ETDataset/1d16c8f4f943005d613b5bc962e9eeb06058cf07/ETT-small/ETTm1.csv (please retain the 69680 chronologically ordered OT observations)
# Weather: https://huggingface.co/datasets/thuml/Time-Series-Library/resolve/2b66e59ee19dac8f6f19fb5d4997f289fdfea357/weather/weather.csv?download=true (please retain OT, remove duplicate timestamps, and provide 52695 ordered rows)
# ECL: https://huggingface.co/datasets/thuml/Time-Series-Library/resolve/2b66e59ee19dac8f6f19fb5d4997f289fdfea357/electricity/electricity.csv?download=true (please standardize each of the 321 client columns with its population mean and standard deviation from the first 18412 rows, average the standardized clients at each timestamp, save the result as OT, and retain 26304 ordered rows)

DATA_LOCATIONS = {
    "JenaClimate": "",
    "AzureLLMTrace": "",
    "ETTh1": "",
    "ETTm1": "",
    "Weather": "",
    "ECL": "",
}

DATA_TARGETS = {
    "JenaClimate": "T (degC)",
    "AzureLLMTrace": "ContextTokens",
    "ETTh1": "OT",
    "ETTm1": "OT",
    "Weather": "OT",
    "ECL": "OT",
}

EXPECTED_ROWS = {
    "JenaClimate": 70041,
    "AzureLLMTrace": 200000,
    "ETTh1": 17420,
    "ETTm1": 69680,
    "Weather": 52695,
    "ECL": 26304,
}

ONE_STEP_DATASETS = ("JenaClimate", "AzureLLMTrace")
MULTI_HORIZON_DATASETS = ("JenaClimate", "AzureLLMTrace", "ETTh1", "ETTm1", "Weather", "ECL")
HORIZONS = (96, 192, 336, 720)
SEED = 42
TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15
INITIAL_TEST_ORIGINS = 256
ONE_STEP_WINDOWS = 256
MULTI_HORIZON_WINDOWS = 8
ONE_STEP_INPUT_LENGTH = 24
MULTI_HORIZON_INPUT_LENGTH = 96
BLOCK_SIZE = 168
MAX_NEW_TOKENS = 4096
MAX_PROMPT_TOKENS = 8192
GENERATION_RETRIES = 2
RANGE_SIGMA = 4.0
RANGE_MARGIN = 1.0
REPORTED_TEMPERATURE = 0.12
REPORTED_TOP_P = 0.90

_DATA_CACHE = {}
_NUMBER_PATTERN = re.compile(r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?")


def set_seed(seed=SEED):
    random.seed(int(seed))
    np.random.seed(int(seed))
    if torch is not None:
        torch.manual_seed(int(seed))
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(int(seed))
        try:
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
        except Exception:
            pass


def configured_models():
    return [name for name, location in MODEL_LOCATIONS.items() if str(location).strip()]


def safe_error(exc):
    text = f"{type(exc).__name__}: {exc}"
    private_values = [HF_TOKEN]
    private_values.extend(str(value) for value in MODEL_LOCATIONS.values())
    private_values.extend(str(value) for value in DATA_LOCATIONS.values())
    for value in private_values:
        if value:
            text = text.replace(value, "<configured_value>")
    return text[:1000]


def infer_date_column(frame):
    accepted = {"date", "ds", "time", "timestamp", "datetime", "date time"}
    for column in frame.columns:
        if str(column).lower() in accepted:
            return column
    return None


def read_jena_frame():
    location = str(DATA_LOCATIONS["JenaClimate"]).strip()
    if location:
        frame = pd.read_csv(Path(location).expanduser())
    else:
        with urllib.request.urlopen(JENA_URL, timeout=180) as response:
            payload = response.read()
        with zipfile.ZipFile(io.BytesIO(payload)) as archive:
            member = next(name for name in archive.namelist() if name.lower().endswith(".csv"))
            with archive.open(member) as stream:
                frame = pd.read_csv(stream, usecols=["Date Time", "T (degC)"])
    date_column = infer_date_column(frame)
    if date_column is not None and len(frame) > EXPECTED_ROWS["JenaClimate"]:
        frame[date_column] = pd.to_datetime(frame[date_column], format="%d.%m.%Y %H:%M:%S", errors="coerce")
        frame = frame.dropna(subset=[date_column]).set_index(date_column).resample("1h")[["T (degC)"]].mean().dropna(subset=["T (degC)"]).reset_index()
    return frame


def read_local_frame(name):
    location = str(DATA_LOCATIONS[name]).strip()
    if not location:
        return None
    path = Path(location).expanduser()
    if not path.is_file():
        raise FileNotFoundError(f"{name} data file was not found")
    return pd.read_csv(path)


def resolve_target(name, frame):
    frame = frame.copy()
    date_column = infer_date_column(frame)
    if date_column is not None:
        timestamps = pd.to_datetime(frame[date_column], errors="raise")
        if timestamps.isna().any() or timestamps.duplicated().any() or not timestamps.is_monotonic_increasing:
            raise ValueError(f"{name} timestamps must be finite, unique, and chronologically ordered")
    requested = DATA_TARGETS[name]
    lowered = {str(column).lower(): column for column in frame.columns}
    target = requested if requested in frame.columns else lowered.get(str(requested).lower())
    if target is None:
        raise ValueError(f"{name} requires a preprocessed {requested} target column")
    try:
        result = pd.to_numeric(frame[target], errors="raise").to_numpy(np.float64)
    except Exception as exc:
        raise ValueError(f"{name} target must contain only numeric values") from exc
    if not np.isfinite(result).all():
        raise ValueError(f"{name} target contains missing or non-finite values")
    if len(result) != EXPECTED_ROWS[name]:
        raise ValueError(f"{name} has {len(result)} rows; expected {EXPECTED_ROWS[name]}")
    return result


def load_dataset(name):
    if name in _DATA_CACHE:
        return _DATA_CACHE[name]
    if name == "JenaClimate":
        frame = read_jena_frame()
    else:
        frame = read_local_frame(name)
        if frame is None:
            return None
    values = resolve_target(name, frame)
    _DATA_CACHE[name] = values
    return values


def evenly_select(values, count):
    values = np.asarray(values, dtype=int)
    if len(values) <= int(count):
        return values
    indices = np.linspace(0, len(values) - 1, int(count)).astype(int)
    return values[indices]


def prepare_windows(values, input_length, horizon, measured_windows):
    values = np.asarray(values, dtype=np.float64).reshape(-1)
    total = len(values)
    train_end = int(total * TRAIN_RATIO)
    validation_end = int(total * (TRAIN_RATIO + VALIDATION_RATIO))
    training_values = values[:train_end]
    mean = float(np.mean(training_values))
    scale = float(np.std(training_values, ddof=0))
    if not np.isfinite(scale) or scale <= 0:
        scale = 1.0
    scaled = ((values - mean) / scale).astype(np.float32)
    valid_origins = np.arange(max(int(input_length), validation_end), max(int(input_length), total - int(horizon) + 1), dtype=int)
    origin_pool = evenly_select(valid_origins, INITIAL_TEST_ORIGINS)
    origins = origin_pool[:int(measured_windows)]
    if len(origins) != int(measured_windows):
        raise ValueError(f"Only {len(origins)} valid test origins are available")
    histories = np.stack([scaled[origin - int(input_length):origin] for origin in origins]).astype(np.float32)
    targets = np.stack([scaled[origin:origin + int(horizon)] for origin in origins]).astype(np.float32)
    return {
        "mean": mean,
        "scale": scale,
        "train_scaled": scaled[:train_end],
        "origins": origins,
        "histories": histories,
        "targets": targets,
    }


class STM:
    def __init__(self, k=5, t_max=96, period_tolerance=1, period_threshold=0.90, period_near_best=0.05, period_min_cycles=3, period_prominence_threshold=4.0):
        self.k = int(k)
        self.t_max = int(t_max)
        self.period_tolerance = int(period_tolerance)
        self.period_threshold = float(period_threshold)
        self.period_near_best = float(period_near_best)
        self.period_min_cycles = int(period_min_cycles)
        self.period_prominence_threshold = float(period_prominence_threshold)
        self.symbols = ["VL", "L", "M", "H", "VH"]
        self.edges = None
        if self.k != 5:
            raise ValueError("The public protocol fixes k at 5")

    def fit(self, training_values):
        values = np.asarray(training_values, dtype=float).reshape(-1)
        values = values[np.isfinite(values)]
        if not len(values):
            raise ValueError("STM requires finite training values")
        low = float(np.min(values))
        high = float(np.max(values))
        if high <= low:
            high = low + 1e-6
        edges = np.linspace(low, high, self.k + 1)
        edges[0] = -np.inf
        edges[-1] = np.inf
        self.edges = edges
        return self

    def transform_codes(self, values):
        if self.edges is None:
            raise RuntimeError("STM must be fitted on the training partition")
        return np.clip(np.digitize(np.asarray(values, dtype=float), self.edges[1:-1]), 0, self.k - 1).astype(int)

    def detect_period(self, deltas):
        deltas = np.asarray(deltas, dtype=int).reshape(-1)
        length = len(deltas)
        if length < 2 * self.period_min_cycles:
            return {"period": None, "score": 0.0, "prominence": 0.0}
        maximum_period = min(self.t_max, length // max(2, self.period_min_cycles))
        scores = {
            period: float(np.mean(np.abs(deltas[:-period] - deltas[period:]) <= self.period_tolerance))
            for period in range(2, maximum_period + 1)
        }
        if not scores:
            return {"period": None, "score": 0.0, "prominence": 0.0}
        all_scores = np.asarray(list(scores.values()), dtype=float)
        prominence = {}
        for period, score in scores.items():
            background = np.asarray([
                value for other, value in scores.items()
                if other % period != 0 and period % other != 0
            ], dtype=float)
            if len(background) < 5:
                background = all_scores
            median = float(np.median(background))
            mad = float(np.median(np.abs(background - median)) + 1e-6)
            prominence[period] = (score - median) / mad
        best = max(scores.values())
        accepted = [
            period for period, score in scores.items()
            if score >= self.period_threshold
            and score >= best - self.period_near_best
            and prominence[period] >= self.period_prominence_threshold
        ]
        period = min(accepted) if accepted else None
        return {
            "period": period,
            "score": float(scores[period] if period is not None else best),
            "prominence": float(prominence[period] if period is not None else max(prominence.values())),
        }

    def compute(self, values):
        codes = self.transform_codes(values)
        deltas = np.diff(codes, prepend=codes[0]).astype(int)
        period_information = self.detect_period(deltas)
        periodic_weight = np.ones(len(codes), dtype=float)
        if period_information["period"] is not None:
            period = int(period_information["period"])
            periodic_weight[np.arange(len(codes)) % period == 0] += float(period_information["score"])
        maximum_symbol_distance = max(1, self.k - 1)
        alpha_raw = (np.abs(deltas).astype(float) / maximum_symbol_distance) * periodic_weight
        alpha_sum = float(np.sum(alpha_raw))
        alpha = alpha_raw / alpha_sum if alpha_sum > 0.0 else np.zeros_like(alpha_raw)
        return {
            "codes": codes,
            "symbols": [self.symbols[code] for code in codes],
            "deltas": deltas,
            "periodic_weight": periodic_weight,
            "alpha_raw": alpha_raw,
            "alpha": alpha,
            "period": period_information["period"],
            "period_score": period_information["score"],
        }


def format_numbers(values, precision=4):
    return ", ".join(f"{float(value):.{int(precision)}f}" for value in np.asarray(values).reshape(-1))


def make_one_step_prompt(sequence, variant, stm, dataset_name):
    numeric = format_numbers(sequence)
    if dataset_name == "JenaClimate":
        task = f"The normalized temperature readings for the past 24 hours are: {numeric}."
        question = "What is the next normalized temperature reading?"
    else:
        task = f"Given the following sequence of normalized inference traffic values: {numeric}."
        question = "Predict the next normalized traffic value based on the pattern and trend."
    symbolic = ""
    if variant == "full_stm":
        symbolic = " STM symbolic pattern: " + " ".join(stm.compute(sequence)["symbols"]) + "."
    elif variant != "raw":
        raise ValueError("variant must be raw or full_stm")
    return task + symbolic + " " + question + " Return exactly one number and no explanation."


def make_representation_prompt(sequence, variant, stm):
    numeric = format_numbers(sequence)
    base = "Encode the observed normalized univariate time series for forecasting. " + f"Numeric sequence: {numeric}. "
    if variant == "raw":
        return base + "Use the numeric level, trend, and recent dynamics."
    if variant != "full_stm":
        raise ValueError("variant must be raw or full_stm")
    representation = stm.compute(sequence)
    pieces = ["Symbolic levels: " + " ".join(representation["symbols"]) + "."]
    pieces.append(
        "Mean absolute symbolic transition: "
        + f"{np.mean(np.abs(representation['deltas'])):.4f}; "
        + "net symbolic direction: "
        + f"{int(np.sum(representation['deltas']))}."
    )
    if representation["period"] is None:
        pieces.append("No accepted repeating interval; best reliability: " + f"{representation['period_score']:.4f}.")
    else:
        pieces.append(
            "Detected repeating interval: "
            + f"{representation['period']}; reliability: "
            + f"{representation['period_score']:.4f}."
        )
    return base + "External symbolic prompt guidance: " + " ".join(pieces)


def make_multi_horizon_prompt(sequence, requested_values, variant, stm):
    representation = make_representation_prompt(sequence, variant, stm)
    return (
        "You are forecasting a normalized univariate time series. "
        + representation
        + f" Forecast exactly the next {int(requested_values)} consecutive normalized values. "
        + "Return only one bracketed comma-separated numeric list, with no labels, indices, units, or explanation."
    )


def parse_generated_numbers(text, requested_values):
    groups = re.findall(r"[\[\(]([^\]\)]{1,20000})[\]\)]", str(text), flags=re.S)
    candidates = groups + [str(text)]
    best = []
    for candidate in candidates:
        parsed = []
        for token in _NUMBER_PATTERN.findall(candidate.replace(",", " ")):
            try:
                parsed.append(float(token))
            except Exception:
                pass
        if len(parsed) > len(best):
            best = parsed
    output = np.full(int(requested_values), np.nan, dtype=float)
    count = min(len(best), int(requested_values))
    if count:
        output[:count] = np.asarray(best[:count], dtype=float)
    return output, int(count)


def symmetric_history_guard(prediction, history):
    prediction = np.asarray(prediction, dtype=float).copy()
    history = np.asarray(history, dtype=float).reshape(-1)
    last_value = float(history[-1])
    prediction = np.where(np.isfinite(prediction), prediction, last_value)
    mean = float(np.mean(history))
    standard_deviation = float(np.std(history, ddof=0) + 1e-6)
    lower = min(float(np.min(history)) - RANGE_MARGIN * standard_deviation, mean - RANGE_SIGMA * standard_deviation)
    upper = max(float(np.max(history)) + RANGE_MARGIN * standard_deviation, mean + RANGE_SIGMA * standard_deviation)
    return np.clip(prediction, lower, upper)


def load_frozen_model(model_name):
    if RUNTIME_ERROR or torch is None or AutoTokenizer is None or AutoModelForCausalLM is None:
        raise RuntimeError("PyTorch and Transformers are required")
    location = str(MODEL_LOCATIONS[model_name]).strip()
    if not location:
        raise ValueError(f"Set MODEL_LOCATIONS for {model_name}")
    revision = str(MODEL_REVISIONS[model_name]).strip() or None
    common = {"token": HF_TOKEN or None, "revision": revision}
    common = {key: value for key, value in common.items() if value is not None}
    tokenizer = None
    tokenizer_error = None
    for trust_remote_code in (False, True):
        try:
            tokenizer = AutoTokenizer.from_pretrained(location, trust_remote_code=trust_remote_code, **common)
            break
        except Exception as exc:
            tokenizer_error = exc
    if tokenizer is None:
        raise RuntimeError(f"Tokenizer loading failed: {tokenizer_error}")
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token or tokenizer.unk_token
    if tokenizer.pad_token_id is None:
        raise RuntimeError("Tokenizer has no usable pad token")
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model = None
    model_error = None
    for trust_remote_code in (False, True):
        base = {"trust_remote_code": trust_remote_code, "low_cpu_mem_usage": True, **common}
        if torch.cuda.is_available():
            base["device_map"] = {"": 0}
        for dtype_key in ("dtype", "torch_dtype"):
            arguments = dict(base)
            arguments[dtype_key] = dtype
            try:
                model = AutoModelForCausalLM.from_pretrained(location, **arguments)
                break
            except TypeError as exc:
                model_error = exc
            except Exception as exc:
                model_error = exc
                break
        if model is not None:
            break
    if model is None:
        raise RuntimeError(f"Model loading failed: {model_error}")
    if not torch.cuda.is_available():
        model.to("cpu")
    model.eval()
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    model._stm_no_cache = False
    if any(parameter.requires_grad for parameter in model.parameters()):
        raise RuntimeError("The backbone was not fully frozen")
    return tokenizer, model


def release_model(tokenizer, model):
    del tokenizer
    del model
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()


def is_cache_error(exc):
    message = str(exc).lower()
    keys = ("dynamiccache", "past_key_values", "cache_position", "seen_tokens", "get_max_length", "sizes of tensors must match", "sliding_window")
    return any(key in message for key in keys)


def generate_completion(tokenizer, model, prompt, maximum_new_tokens):
    messages = [{"role": "user", "content": prompt}]
    try:
        rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        rendered = prompt
    encoded = tokenizer(rendered, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_TOKENS)
    encoded = {key: value.to(model.device) for key, value in encoded.items()}
    arguments = {
        "max_new_tokens": int(maximum_new_tokens),
        "do_sample": False,
        "num_return_sequences": 1,
        "pad_token_id": tokenizer.pad_token_id,
        "use_cache": not bool(getattr(model, "_stm_no_cache", False)),
    }
    try:
        with torch.inference_mode():
            output = model.generate(**encoded, **arguments)
    except Exception as exc:
        if arguments["use_cache"] and is_cache_error(exc):
            model._stm_no_cache = True
            arguments["use_cache"] = False
            with torch.inference_mode():
                output = model.generate(**encoded, **arguments)
        else:
            raise
    generated = output[0, encoded["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


def forecast_one_step(tokenizer, model, history, variant, stm, dataset_name):
    prompt = make_one_step_prompt(history, variant, stm, dataset_name)
    completion = generate_completion(tokenizer, model, prompt, 64)
    values, count = parse_generated_numbers(completion, 1)
    if not count or not np.isfinite(values[0]):
        values[0] = float(np.asarray(history).reshape(-1)[-1])
    return float(values[0]), int(count)


def forecast_block(tokenizer, model, history, count, variant, stm):
    prompt = make_multi_horizon_prompt(history, count, variant, stm)
    best = np.full(int(count), np.nan, dtype=float)
    best_count = 0
    budget = min(MAX_NEW_TOKENS, max(64, 12 * int(count) + 32))
    for attempt in range(GENERATION_RETRIES):
        current_prompt = prompt if attempt == 0 else prompt + " Your previous format was incomplete; output the complete numeric list now."
        completion = generate_completion(tokenizer, model, current_prompt, budget)
        values, parsed_count = parse_generated_numbers(completion, count)
        if parsed_count > best_count:
            best = values
            best_count = parsed_count
        if parsed_count >= int(count):
            break
    return best, int(best_count)


def forecast_multi_horizon(tokenizer, model, input_sequence, horizon, variant, stm):
    context = list(np.asarray(input_sequence, dtype=float).reshape(-1))
    predictions = []
    parsed = 0
    requested = 0
    while len(predictions) < int(horizon):
        count = min(BLOCK_SIZE, int(horizon) - len(predictions))
        recent_history = np.asarray(context[-MULTI_HORIZON_INPUT_LENGTH:], dtype=float)
        values, parsed_count = forecast_block(tokenizer, model, recent_history, count, variant, stm)
        values = symmetric_history_guard(values, recent_history)
        predictions.extend(values.tolist())
        context.extend(values.tolist())
        parsed += min(parsed_count, count)
        requested += count
    return np.asarray(predictions[:int(horizon)], dtype=float), parsed / max(1, requested)


def inverse_scale(values, mean, scale):
    return np.asarray(values, dtype=float) * float(scale) + float(mean)


def core_self_test():
    stm = STM().fit(np.linspace(-2.0, 2.0, 101))
    codes = stm.transform_codes(np.array([-3.0, -1.0, 0.0, 1.0, 3.0]))
    periodic = stm.detect_period(np.tile(np.array([2, -1, 0, 1]), 12))
    parsed, count = parse_generated_numbers("[-1.25, 2.5e-1]", 2)
    raw = make_representation_prompt(np.linspace(-1.0, 1.0, 8), "raw", stm)
    full = make_representation_prompt(np.linspace(-1.0, 1.0, 8), "full_stm", stm)
    numeric = "Numeric sequence: " + format_numbers(np.linspace(-1.0, 1.0, 8)) + ". "
    passed = bool(
        codes.min() == 0
        and codes.max() == 4
        and periodic["period"] == 4
        and count == 2
        and np.allclose(parsed, [-1.25, 0.25])
        and numeric in raw
        and numeric in full
    )
    return "passed" if passed else "failed"


set_seed()
print("result:", {
    "status": "ready" if not RUNTIME_ERROR else "runtime_configuration_required",
    "runtime": "ready" if not RUNTIME_ERROR else RUNTIME_ERROR,
    "required_packages": REQUIRED_PACKAGES,
    "required_models": list(MODEL_LOCATIONS),
    "configured_models": configured_models(),
    "jena_source": JENA_URL,
    "local_datasets_to_add": [name for name in DATA_LOCATIONS if name != "JenaClimate"],
    "core_self_test": core_self_test(),
})

In [ ]:
def run_one_step_experiment():
    models = configured_models()
    if not models:
        return {
            "status": "configuration_required",
            "required_models": list(MODEL_LOCATIONS),
            "required_datasets": list(ONE_STEP_DATASETS),
        }
    bundles = {}
    missing_datasets = []
    errors = []
    for dataset_name in ONE_STEP_DATASETS:
        try:
            values = load_dataset(dataset_name)
            if values is None:
                missing_datasets.append(dataset_name)
                continue
            bundles[dataset_name] = prepare_windows(values, ONE_STEP_INPUT_LENGTH, 1, ONE_STEP_WINDOWS)
        except Exception as exc:
            errors.append({"dataset": dataset_name, "error": safe_error(exc)})
    if not bundles:
        return {
            "status": "configuration_required",
            "required_models": list(MODEL_LOCATIONS),
            "required_datasets": list(ONE_STEP_DATASETS),
            "missing_datasets": missing_datasets,
            "errors": errors,
        }
    results = []
    for model_name in models:
        tokenizer = None
        model = None
        try:
            tokenizer, model = load_frozen_model(model_name)
            for dataset_name, bundle in bundles.items():
                stm = STM().fit(bundle["train_scaled"])
                for variant in ("raw", "full_stm"):
                    predictions = []
                    parsed = []
                    for history in bundle["histories"]:
                        prediction, parsed_count = forecast_one_step(tokenizer, model, history, variant, stm, dataset_name)
                        predictions.append(prediction)
                        parsed.append(parsed_count)
                    predictions = np.asarray(predictions, dtype=float)
                    targets = bundle["targets"][:, 0]
                    predictions_original = inverse_scale(predictions, bundle["mean"], bundle["scale"])
                    targets_original = inverse_scale(targets, bundle["mean"], bundle["scale"])
                    results.append({
                        "dataset": dataset_name,
                        "model": model_name,
                        "variant": variant,
                        "test_origins": int(len(targets)),
                        "MAE": round(float(np.mean(np.abs(targets_original - predictions_original))), 8),
                        "MSE": round(float(np.mean((targets_original - predictions_original) ** 2)), 8),
                        "parsed_fraction": round(float(np.mean(parsed)), 8),
                    })
        except Exception as exc:
            errors.append({"model": model_name, "error": safe_error(exc)})
        finally:
            if tokenizer is not None and model is not None:
                release_model(tokenizer, model)
                tokenizer = None
                model = None
    return {
        "status": "ok" if results and not errors and not missing_datasets else "partial" if results else "failed",
        "protocol": {
            "input_length": ONE_STEP_INPUT_LENGTH,
            "horizon": 1,
            "test_origins": ONE_STEP_WINDOWS,
            "do_sample": False,
            "temperature_reported": REPORTED_TEMPERATURE,
            "top_p_reported": REPORTED_TOP_P,
        },
        "results": results,
        "missing_datasets": missing_datasets,
        "errors": errors,
    }


print("result:", run_one_step_experiment())

In [ ]:
def run_multi_horizon_experiment():
    models = configured_models()
    if not models:
        return {
            "status": "configuration_required",
            "required_models": list(MODEL_LOCATIONS),
            "required_datasets": list(MULTI_HORIZON_DATASETS),
        }
    bundles = {}
    missing_datasets = []
    errors = []
    for dataset_name in MULTI_HORIZON_DATASETS:
        try:
            values = load_dataset(dataset_name)
            if values is None:
                missing_datasets.append(dataset_name)
                continue
            for horizon in HORIZONS:
                bundles[(dataset_name, int(horizon))] = prepare_windows(
                    values,
                    MULTI_HORIZON_INPUT_LENGTH,
                    int(horizon),
                    MULTI_HORIZON_WINDOWS,
                )
        except Exception as exc:
            errors.append({"dataset": dataset_name, "error": safe_error(exc)})
    if not bundles:
        return {
            "status": "configuration_required",
            "required_models": list(MODEL_LOCATIONS),
            "required_datasets": list(MULTI_HORIZON_DATASETS),
            "missing_datasets": missing_datasets,
            "errors": errors,
        }
    results = []
    for model_name in models:
        tokenizer = None
        model = None
        try:
            tokenizer, model = load_frozen_model(model_name)
            for (dataset_name, horizon), bundle in bundles.items():
                stm = STM().fit(bundle["train_scaled"])
                for variant in ("raw", "full_stm"):
                    window_nmae = []
                    parsed_fractions = []
                    for history, target in zip(bundle["histories"], bundle["targets"]):
                        prediction, parsed_fraction = forecast_multi_horizon(tokenizer, model, history, horizon, variant, stm)
                        window_nmae.append(float(np.mean(np.abs(np.asarray(target, dtype=float) - prediction))))
                        parsed_fractions.append(float(parsed_fraction))
                    results.append({
                        "dataset": dataset_name,
                        "horizon": int(horizon),
                        "model": model_name,
                        "variant": variant,
                        "test_origins": int(len(window_nmae)),
                        "NMAE_mean": round(float(np.mean(window_nmae)), 8),
                        "NMAE_std": round(float(np.std(window_nmae, ddof=1)), 8),
                        "parsed_fraction": round(float(np.mean(parsed_fractions)), 8),
                    })
        except Exception as exc:
            errors.append({"model": model_name, "error": safe_error(exc)})
        finally:
            if tokenizer is not None and model is not None:
                release_model(tokenizer, model)
                tokenizer = None
                model = None
    return {
        "status": "ok" if results and not errors and not missing_datasets else "partial" if results else "failed",
        "protocol": {
            "input_length": MULTI_HORIZON_INPUT_LENGTH,
            "horizons": HORIZONS,
            "test_origins": MULTI_HORIZON_WINDOWS,
            "initial_origin_pool": INITIAL_TEST_ORIGINS,
            "block_size": BLOCK_SIZE,
            "do_sample": False,
            "temperature_reported": REPORTED_TEMPERATURE,
            "top_p_reported": REPORTED_TOP_P,
        },
        "results": results,
        "missing_datasets": missing_datasets,
        "errors": errors,
    }


print("result:", run_multi_horizon_experiment())

In [ ]:
import json
import os
import platform
import shutil
import socket
import subprocess
import sys
import tempfile
import threading
import time
import urllib.request
from pathlib import Path

RESOURCE_RUNTIME_ERROR = ""
try:
    import psutil
except Exception as exc:
    psutil = None
    RESOURCE_RUNTIME_ERROR = f"{type(exc).__name__}: {exc}"

LLAMA_SERVER_PATH = ""

GGUF_MODEL_LOCATIONS = {
    "Phi-3.5-mini": "",
    "LLaMA-3.2-1B": "",
    "DeepSeek-1.3B": "",
    "Gemma-2B-it": "",
}

CHAT_TEMPLATE_LOCATIONS = {
    "Phi-3.5-mini": "",
    "LLaMA-3.2-1B": "",
    "DeepSeek-1.3B": "",
    "Gemma-2B-it": "",
}

RESOURCE_ALIASES = {
    "Phi-3.5-mini": "stm-phi35-q4km",
    "LLaMA-3.2-1B": "stm-llama32-1b-q4km",
    "DeepSeek-1.3B": "stm-deepseek13-q4km",
    "Gemma-2B-it": "stm-gemma2b-q4km",
}

RESOURCE_LLAMA_CPP_BUILD = "10287"
RESOURCE_LLAMA_CPP_COMMIT = "b06aa77"
RESOURCE_CPU_THREADS = 4
RESOURCE_MEMORY_LIMIT_BYTES = int(3.5 * 1024**3)
RESOURCE_WINDOWS = 8
RESOURCE_REPEATS = 3
RESOURCE_INPUT_LENGTH = 24
RESOURCE_CONTEXT_SIZE = 512
RESOURCE_BATCH_SIZE = 256
RESOURCE_UBATCH_SIZE = 128
RESOURCE_OUTPUT_TOKENS = 64
RESOURCE_TIMEOUT_SECONDS = 300
RESOURCE_START_TIMEOUT_SECONDS = 240
RESOURCE_SCHEMA = {
    "type": "array",
    "minItems": 1,
    "maxItems": 1,
    "items": {"type": "number"},
}

RESOURCE_WINDOWS_LAUNCHER = """
import json
import os
import subprocess
import sys
import time
gate = os.environ["STM_RESOURCE_GATE"]
command = json.loads(os.environ["STM_RESOURCE_COMMAND"])
working_directory = os.environ["STM_RESOURCE_CWD"]
while not os.path.exists(gate):
    time.sleep(0.01)
child = subprocess.Popen(command, cwd=working_directory, stdout=sys.stdout, stderr=subprocess.STDOUT)
try:
    raise SystemExit(child.wait())
finally:
    if child.poll() is None:
        child.terminate()
"""

RESOURCE_OPENER = urllib.request.build_opener(urllib.request.ProxyHandler({}))


def resource_safe_error(exc):
    message = f"{type(exc).__name__}: {exc}"
    private_values = [LLAMA_SERVER_PATH]
    private_values.extend(str(value) for value in GGUF_MODEL_LOCATIONS.values())
    private_values.extend(str(value) for value in CHAT_TEMPLATE_LOCATIONS.values())
    private_values.extend(str(value) for value in DATA_LOCATIONS.values())
    for value in private_values:
        if value:
            message = message.replace(value, "<configured_value>")
    return message[:1000]


def resource_post_json(url, payload=None, timeout=RESOURCE_TIMEOUT_SECONDS):
    body = None if payload is None else json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(
        url,
        data=body,
        headers={"Content-Type": "application/json", "Authorization": "Bearer no-key"},
        method="GET" if payload is None else "POST",
    )
    with RESOURCE_OPENER.open(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


def resource_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(("127.0.0.1", 0))
        return int(sock.getsockname()[1])


def resource_rss_tree_bytes(pid):
    if psutil is None:
        return 0
    try:
        process = psutil.Process(pid)
        total = process.memory_info().rss
        for child in process.children(recursive=True):
            try:
                total += child.memory_info().rss
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                pass
        return int(total)
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        return 0


def resource_affinity_cpus():
    if psutil is None:
        return []
    try:
        available = list(psutil.Process().cpu_affinity())
    except Exception:
        available = list(range(psutil.cpu_count(logical=True) or 0))
    return available[:RESOURCE_CPU_THREADS]


def resource_assign_windows_job(process, cpus):
    import win32api
    import win32con
    import win32job

    job = win32job.CreateJobObject(None, "")
    information = win32job.QueryInformationJobObject(job, win32job.JobObjectExtendedLimitInformation)
    information["BasicLimitInformation"]["LimitFlags"] |= (
        win32job.JOB_OBJECT_LIMIT_PROCESS_MEMORY
        | win32job.JOB_OBJECT_LIMIT_JOB_MEMORY
        | win32job.JOB_OBJECT_LIMIT_KILL_ON_JOB_CLOSE
    )
    information["ProcessMemoryLimit"] = int(RESOURCE_MEMORY_LIMIT_BYTES)
    information["JobMemoryLimit"] = int(RESOURCE_MEMORY_LIMIT_BYTES)
    win32job.SetInformationJobObject(job, win32job.JobObjectExtendedLimitInformation, information)
    observed = win32job.QueryInformationJobObject(job, win32job.JobObjectExtendedLimitInformation)
    required_flags = (
        win32job.JOB_OBJECT_LIMIT_PROCESS_MEMORY
        | win32job.JOB_OBJECT_LIMIT_JOB_MEMORY
        | win32job.JOB_OBJECT_LIMIT_KILL_ON_JOB_CLOSE
    )
    observed_flags = int(observed["BasicLimitInformation"]["LimitFlags"])
    if observed_flags & required_flags != required_flags:
        raise RuntimeError("Windows Job Object limit flags did not persist")
    if int(observed.get("ProcessMemoryLimit", -1)) != RESOURCE_MEMORY_LIMIT_BYTES:
        raise RuntimeError("Windows process-memory cap read-back mismatch")
    if int(observed.get("JobMemoryLimit", -1)) != RESOURCE_MEMORY_LIMIT_BYTES:
        raise RuntimeError("Windows job-memory cap read-back mismatch")
    handle = win32api.OpenProcess(
        win32con.PROCESS_SET_QUOTA
        | win32con.PROCESS_TERMINATE
        | getattr(win32con, "PROCESS_QUERY_LIMITED_INFORMATION", 0x1000),
        False,
        process.pid,
    )
    win32job.AssignProcessToJobObject(job, handle)
    if not win32job.IsProcessInJob(handle, job):
        raise RuntimeError("The Windows process was not assigned to the memory-capped Job Object")
    psutil.Process(process.pid).cpu_affinity(cpus)
    return job, handle


def resource_configuration_errors():
    errors = []
    if RESOURCE_RUNTIME_ERROR:
        errors.append(RESOURCE_RUNTIME_ERROR)
    system = platform.system()
    if system not in {"Windows", "Linux"}:
        errors.append("The hard memory cap is implemented for Windows and Linux")
    if system == "Windows":
        try:
            import win32api
            import win32con
            import win32job
        except Exception as exc:
            errors.append(f"pywin32 is required: {type(exc).__name__}: {exc}")
    if system == "Linux" and (not shutil.which("prlimit") or not shutil.which("taskset")):
        errors.append("Linux requires prlimit and taskset")
    server = Path(str(LLAMA_SERVER_PATH).strip()).expanduser() if str(LLAMA_SERVER_PATH).strip() else None
    if server is None or not server.is_file():
        errors.append("Set LLAMA_SERVER_PATH to the pinned llama-server executable")
    missing_models = [name for name, path in GGUF_MODEL_LOCATIONS.items() if not str(path).strip() or not Path(str(path)).expanduser().is_file()]
    if missing_models:
        errors.append("Set Q4_K_M GGUF paths for: " + ", ".join(missing_models))
    invalid_quantization = [name for name, path in GGUF_MODEL_LOCATIONS.items() if str(path).strip() and "q4_k_m" not in Path(str(path)).expanduser().name.lower()]
    if invalid_quantization:
        errors.append("Q4_K_M filenames are required for: " + ", ".join(invalid_quantization))
    invalid_templates = [name for name, path in CHAT_TEMPLATE_LOCATIONS.items() if str(path).strip() and not Path(str(path)).expanduser().is_file()]
    if invalid_templates:
        errors.append("Chat-template files were not found for: " + ", ".join(invalid_templates))
    deepseek_template = str(CHAT_TEMPLATE_LOCATIONS["DeepSeek-1.3B"]).strip()
    if not deepseek_template or not Path(deepseek_template).expanduser().is_file():
        errors.append("Set the DeepSeek-1.3B chat-template file path")
    if len(resource_affinity_cpus()) != RESOURCE_CPU_THREADS:
        errors.append("Four logical CPUs are required")
    if server is not None and server.is_file():
        try:
            version = subprocess.run(
                [str(server), "--version"],
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                timeout=30,
            )
            observed = version.stdout or ""
            if version.returncode != 0 or RESOURCE_LLAMA_CPP_BUILD not in observed or RESOURCE_LLAMA_CPP_COMMIT not in observed:
                errors.append("llama-server build 10287 commit b06aa77 is required")
        except Exception as exc:
            errors.append(resource_safe_error(exc))
    missing_data = [name for name in MULTI_HORIZON_DATASETS if name != "JenaClimate" and not str(DATA_LOCATIONS[name]).strip()]
    if missing_data:
        errors.append("Set preprocessed local CSV paths for: " + ", ".join(missing_data))
    invalid_data = [name for name in MULTI_HORIZON_DATASETS if str(DATA_LOCATIONS[name]).strip() and not Path(str(DATA_LOCATIONS[name])).expanduser().is_file()]
    if invalid_data:
        errors.append("Configured dataset files were not found for: " + ", ".join(invalid_data))
    return errors


class ResourceServer:
    def __init__(self, model_name):
        self.model_name = model_name
        self.model_path = Path(GGUF_MODEL_LOCATIONS[model_name]).expanduser().resolve()
        template = str(CHAT_TEMPLATE_LOCATIONS[model_name]).strip()
        self.template_path = Path(template).expanduser().resolve() if template else None
        self.port = resource_free_port()
        self.base_url = f"http://127.0.0.1:{self.port}"
        self.process = None
        self.job_handle = None
        self.process_handle = None
        self.temporary_directory = None
        self.gate_path = None

    def command(self, slot_directory):
        command = [
            str(Path(LLAMA_SERVER_PATH).expanduser().resolve()),
            "-m",
            str(self.model_path),
            "--alias",
            RESOURCE_ALIASES[self.model_name],
            "--host",
            "127.0.0.1",
            "--port",
            str(self.port),
            "-t",
            str(RESOURCE_CPU_THREADS),
            "-tb",
            str(RESOURCE_CPU_THREADS),
            "-c",
            str(RESOURCE_CONTEXT_SIZE),
            "-b",
            str(RESOURCE_BATCH_SIZE),
            "-ub",
            str(RESOURCE_UBATCH_SIZE),
            "-np",
            "1",
            "-ngl",
            "0",
            "--device",
            "none",
            "--no-kv-offload",
            "--no-op-offload",
            "--fit",
            "off",
            "--load-mode",
            "none",
            "-ctk",
            "f16",
            "-ctv",
            "f16",
            "--flash-attn",
            "off",
            "--no-cache-prompt",
            "-cram",
            "0",
            "-ctxcp",
            "0",
            "--no-cache-idle-slots",
            "--slot-save-path",
            str(slot_directory),
            "--no-warmup",
            "--jinja",
            "--reasoning-format",
            "none",
            "--threads-http",
            "1",
            "--log-colors",
            "off",
            "--slots",
            "--perf",
        ]
        if self.template_path is not None:
            command.extend(["--chat-template-file", str(self.template_path)])
        return command

    def start(self):
        try:
            self.temporary_directory = tempfile.TemporaryDirectory(prefix="stm_resource_")
            root = Path(self.temporary_directory.name)
            slot_directory = root / "slots"
            slot_directory.mkdir()
            self.gate_path = root / "server.ready"
            command = self.command(slot_directory)
            environment = os.environ.copy()
            environment.update({
                "CUDA_VISIBLE_DEVICES": "",
                "HIP_VISIBLE_DEVICES": "",
                "ROCR_VISIBLE_DEVICES": "",
                "GGML_VK_VISIBLE_DEVICES": "",
                "GGML_CUDA_NO_PINNED": "1",
            })
            arguments = {
                "cwd": str(Path(LLAMA_SERVER_PATH).expanduser().resolve().parent),
                "env": environment,
                "stdout": subprocess.DEVNULL,
                "stderr": subprocess.STDOUT,
                "text": True,
            }
            launch_command = command
            cpus = resource_affinity_cpus()
            if platform.system() == "Linux":
                launch_command = [
                    shutil.which("prlimit"),
                    f"--as={RESOURCE_MEMORY_LIMIT_BYTES}",
                    "--",
                    shutil.which("taskset"),
                    "-c",
                    ",".join(map(str, cpus)),
                    *command,
                ]
                arguments["start_new_session"] = True
            else:
                environment["STM_RESOURCE_GATE"] = str(self.gate_path)
                environment["STM_RESOURCE_COMMAND"] = json.dumps(command)
                environment["STM_RESOURCE_CWD"] = str(Path(LLAMA_SERVER_PATH).expanduser().resolve().parent)
                arguments["creationflags"] = getattr(subprocess, "CREATE_NO_WINDOW", 0)
                launch_command = [sys.executable, "-u", "-c", RESOURCE_WINDOWS_LAUNCHER]
            self.process = subprocess.Popen(launch_command, **arguments)
            if platform.system() == "Windows":
                self.job_handle, self.process_handle = resource_assign_windows_job(self.process, cpus)
                self.gate_path.touch()
            deadline = time.time() + RESOURCE_START_TIMEOUT_SECONDS
            health = None
            while time.time() < deadline:
                if self.process.poll() is not None:
                    break
                try:
                    health = resource_post_json(self.base_url + "/health", timeout=3)
                    if health.get("status") == "ok":
                        break
                except Exception:
                    time.sleep(0.5)
            if health is None or health.get("status") != "ok":
                raise RuntimeError(f"{self.model_name} failed to start under the 3.5 GiB cap")
            if platform.system() == "Linux":
                import resource
                soft, hard = resource.prlimit(self.process.pid, resource.RLIMIT_AS)
                if int(soft) != RESOURCE_MEMORY_LIMIT_BYTES or int(hard) != RESOURCE_MEMORY_LIMIT_BYTES:
                    raise RuntimeError("Linux address-space cap read-back mismatch")
                observed_cpus = list(psutil.Process(self.process.pid).cpu_affinity())
                if observed_cpus != cpus:
                    raise RuntimeError(f"Linux CPU affinity mismatch: {observed_cpus}")
            else:
                descendants = psutil.Process(self.process.pid).children(recursive=True)
                if not descendants:
                    raise RuntimeError("The Windows launcher has no llama-server child")
                import win32api
                import win32con
                import win32job
                for child in descendants:
                    observed_cpus = list(child.cpu_affinity())
                    if observed_cpus != cpus:
                        raise RuntimeError(f"Windows child CPU affinity mismatch: {observed_cpus}")
                    child_handle = win32api.OpenProcess(
                        getattr(win32con, "PROCESS_QUERY_LIMITED_INFORMATION", 0x1000),
                        False,
                        child.pid,
                    )
                    try:
                        if not win32job.IsProcessInJob(child_handle, self.job_handle):
                            raise RuntimeError("The Windows llama-server child escaped the hard-capped Job Object")
                    finally:
                        child_handle.Close()
            props = resource_post_json(self.base_url + "/props", timeout=30)
            effective_template = props.get("chat_template")
            if not isinstance(effective_template, str) or not effective_template.strip():
                raise RuntimeError("llama-server did not expose a nonempty chat template")
            if self.template_path is not None:
                canonical_template = self.template_path.read_text(encoding="utf-8").replace("\r\n", "\n").replace("\r", "\n")
                if canonical_template.endswith("\n"):
                    canonical_template = canonical_template[:-1]
                if effective_template != canonical_template:
                    raise RuntimeError("The effective llama-server chat template does not match the configured file")
            models = resource_post_json(self.base_url + "/v1/models", timeout=30)
            observed = [str(item.get("id")) for item in models.get("data", [])]
            if RESOURCE_ALIASES[self.model_name] not in observed:
                raise RuntimeError(f"Unexpected llama-server model alias for {self.model_name}")
            return self
        except Exception:
            self.close()
            raise

    def prepare(self, prompt):
        rendered = resource_post_json(
            self.base_url + "/apply-template",
            {"messages": [{"role": "user", "content": prompt}]},
            timeout=30,
        ).get("prompt")
        if not isinstance(rendered, str) or not rendered:
            raise RuntimeError("llama-server returned an empty rendered prompt")
        tokenized = resource_post_json(
            self.base_url + "/tokenize",
            {
                "content": rendered,
                "add_special": self.model_name == "DeepSeek-1.3B",
                "parse_special": True,
            },
            timeout=30,
        )
        tokens = tokenized.get("tokens")
        if not isinstance(tokens, list) or not tokens or any(isinstance(token, bool) or not isinstance(token, int) for token in tokens):
            raise RuntimeError("llama-server returned invalid prompt tokens")
        if len(tokens) + RESOURCE_OUTPUT_TOKENS > RESOURCE_CONTEXT_SIZE:
            raise RuntimeError("The rendered prompt exceeds the fixed context allocation")
        return tokens

    def complete(self, tokens, monitor_memory=False):
        erased = resource_post_json(self.base_url + "/slots/0?action=erase", {}, timeout=30)
        if int(erased.get("id_slot", -1)) != 0:
            raise RuntimeError("Could not erase llama-server slot 0")
        stop = threading.Event()
        peak = resource_rss_tree_bytes(self.process.pid)

        def monitor():
            nonlocal peak
            while not stop.is_set():
                peak = max(peak, resource_rss_tree_bytes(self.process.pid))
                time.sleep(0.005)
            peak = max(peak, resource_rss_tree_bytes(self.process.pid))

        thread = None
        if monitor_memory:
            thread = threading.Thread(target=monitor, daemon=True)
            thread.start()
        try:
            response = resource_post_json(
                self.base_url + "/completion",
                {
                    "prompt": tokens,
                    "n_predict": RESOURCE_OUTPUT_TOKENS,
                    "temperature": 0.0,
                    "seed": SEED,
                    "stream": False,
                    "cache_prompt": False,
                    "ignore_eos": False,
                    "json_schema": RESOURCE_SCHEMA,
                    "id_slot": 0,
                },
            )
        finally:
            stop.set()
            if thread is not None:
                thread.join(timeout=2)
        if response.get("truncated"):
            raise RuntimeError("The prompt was truncated")
        hit_limit = bool(response.get("stopped_limit")) or str(response.get("stop_type", "")).lower() == "limit"
        try:
            parsed = json.loads(str(response.get("content", "")).strip())
        except Exception as exc:
            raise RuntimeError("The completion was not a JSON numeric array") from exc
        if not isinstance(parsed, list) or len(parsed) != 1 or isinstance(parsed[0], bool) or not isinstance(parsed[0], (int, float)) or not np.isfinite(float(parsed[0])):
            raise RuntimeError("The completion did not contain exactly one finite forecast")
        timings = response.get("timings") or {}
        required = ("prompt_n", "prompt_ms", "predicted_n", "predicted_ms", "cache_n")
        if hit_limit or any(field not in timings for field in required):
            raise RuntimeError("The completion did not terminate under the nonbinding output ceiling")
        if int(timings["cache_n"]) != 0 or int(timings["prompt_n"]) != len(tokens):
            raise RuntimeError("Prompt caching or token-count mismatch was detected")
        prompt_ms = float(timings["prompt_ms"])
        predicted_ms = float(timings["predicted_ms"])
        predicted_tokens = int(timings["predicted_n"])
        if not np.isfinite(prompt_ms) or not np.isfinite(predicted_ms) or prompt_ms < 0 or predicted_ms <= 0:
            raise RuntimeError("llama-server returned invalid inference timings")
        if predicted_tokens <= 0 or predicted_tokens >= RESOURCE_OUTPUT_TOKENS:
            raise RuntimeError("Unexpected generated-token count")
        if monitor_memory and peak <= 0:
            raise RuntimeError("The server RSS tree could not be measured")
        return {
            "latency_ms": prompt_ms + predicted_ms,
            "peak_rss_bytes": int(peak) if monitor_memory else None,
        }

    def close(self):
        if self.job_handle is not None:
            try:
                self.job_handle.Close()
            except Exception:
                pass
            self.job_handle = None
        if self.process is not None and self.process.poll() is None:
            try:
                self.process.terminate()
                self.process.wait(timeout=10)
            except Exception:
                try:
                    self.process.kill()
                    self.process.wait(timeout=10)
                except Exception:
                    pass
        if self.process_handle is not None:
            try:
                self.process_handle.Close()
            except Exception:
                pass
            self.process_handle = None
        if self.temporary_directory is not None:
            try:
                self.temporary_directory.cleanup()
            except Exception:
                pass
            self.temporary_directory = None

    def __enter__(self):
        return self.start()

    def __exit__(self, exc_type, exc_value, traceback):
        self.close()


def build_resource_prompts():
    prompts = {}
    warmups = {}
    missing = []
    for dataset_name in MULTI_HORIZON_DATASETS:
        values = load_dataset(dataset_name)
        if values is None:
            missing.append(dataset_name)
            continue
        total = len(values)
        train_end = int(total * TRAIN_RATIO)
        validation_end = int(total * (TRAIN_RATIO + VALIDATION_RATIO))
        mean = float(np.mean(values[:train_end]))
        scale = float(np.std(values[:train_end], ddof=0))
        if not np.isfinite(scale) or scale <= 0:
            raise ValueError(f"{dataset_name} has an invalid training scale")
        scaled = ((values - mean) / scale).astype(np.float32)
        candidates = np.arange(max(RESOURCE_INPUT_LENGTH, validation_end), total, dtype=int)
        origins = evenly_select(candidates, RESOURCE_WINDOWS)
        if len(origins) != RESOURCE_WINDOWS:
            raise ValueError(f"{dataset_name} does not provide eight resource-evaluation origins")
        warmup_candidates = np.setdiff1d(candidates, origins, assume_unique=True)
        warmup_origin = int(warmup_candidates[len(warmup_candidates) // 2])
        stm = STM().fit(scaled[:train_end])
        for window_id, origin in enumerate(origins):
            history = scaled[int(origin) - RESOURCE_INPUT_LENGTH:int(origin)]
            for variant in ("raw", "full_stm"):
                prompts[(dataset_name, window_id, variant)] = make_multi_horizon_prompt(history, 1, variant, stm)
        warmup_history = scaled[warmup_origin - RESOURCE_INPUT_LENGTH:warmup_origin]
        for variant in ("raw", "full_stm"):
            warmups[(dataset_name, variant)] = make_multi_horizon_prompt(warmup_history, 1, variant, stm)
    return prompts, warmups, missing


def prepare_resource_tokens(server, prompts, warmups):
    prepared = {key: server.prepare(prompt) for key, prompt in prompts.items()}
    prepared_warmups = {key: server.prepare(prompt) for key, prompt in warmups.items()}
    return prepared, prepared_warmups


def summarize_resource_timing(rows):
    frame = pd.DataFrame(rows)
    required_columns = {"backbone", "dataset", "window_id", "repeat", "variant", "latency_ms"}
    if frame.empty or not required_columns.issubset(frame.columns):
        raise ValueError("Resource timing rows are incomplete")
    key_columns = ["backbone", "dataset", "window_id", "repeat", "variant"]
    if frame.duplicated(key_columns).any():
        raise ValueError("Duplicate resource timing keys were detected")
    frame["latency_ms"] = pd.to_numeric(frame["latency_ms"], errors="raise")
    if not np.isfinite(frame["latency_ms"].to_numpy(dtype=float)).all() or (frame["latency_ms"] <= 0).any():
        raise ValueError("Resource timings must be finite and positive")
    expected_datasets = set(MULTI_HORIZON_DATASETS)
    expected_windows = set(range(RESOURCE_WINDOWS))
    expected_repeats = set(range(RESOURCE_REPEATS))
    expected_variants = {"raw", "full_stm"}
    for backbone, backbone_frame in frame.groupby("backbone", sort=False):
        if set(backbone_frame["dataset"]) != expected_datasets:
            raise ValueError(f"{backbone} does not contain all six dataset configurations")
        if len(backbone_frame) != len(expected_datasets) * RESOURCE_WINDOWS * RESOURCE_REPEATS * len(expected_variants):
            raise ValueError(f"{backbone} does not contain the complete timing design")
        for dataset, dataset_frame in backbone_frame.groupby("dataset", sort=False):
            if set(int(value) for value in dataset_frame["window_id"]) != expected_windows:
                raise ValueError(f"{backbone}/{dataset} does not contain eight windows")
            if set(dataset_frame["variant"]) != expected_variants:
                raise ValueError(f"{backbone}/{dataset} does not contain both prompt variants")
            for _, group in dataset_frame.groupby(["window_id", "variant"], sort=False):
                if set(int(value) for value in group["repeat"]) != expected_repeats:
                    raise ValueError(f"{backbone}/{dataset} does not contain three paired repeats")
    window_means = frame.groupby(["backbone", "dataset", "window_id", "variant"], as_index=False)["latency_ms"].mean()
    paired = window_means.pivot(index=["backbone", "dataset", "window_id"], columns="variant", values="latency_ms").reset_index()
    if not expected_variants.issubset(paired.columns) or paired[list(expected_variants)].isna().any().any():
        raise ValueError("Raw and Full STM timings could not be paired")
    paired["latency_ratio"] = paired["full_stm"] / paired["raw"]
    if not np.isfinite(paired["latency_ratio"].to_numpy(dtype=float)).all() or (paired["latency_ratio"] <= 0).any():
        raise ValueError("Resource latency ratios must be finite and positive")
    configurations = paired.groupby(["backbone", "dataset"], as_index=False)["latency_ratio"].mean()
    configuration_counts = configurations.groupby("backbone").size()
    if (configuration_counts != len(expected_datasets)).any():
        raise ValueError("Each backbone must contain six dataset-level ratios")
    backbone_rows = []
    for backbone, group in configurations.groupby("backbone", sort=False):
        backbone_rows.append({
            "backbone": backbone,
            "latency_ratio_mean": round(float(group["latency_ratio"].mean()), 6),
            "latency_ratio_std": round(float(group["latency_ratio"].std(ddof=1)), 6),
            "dataset_configurations": int(len(group)),
        })
    return backbone_rows, {
        "latency_ratio_mean": round(float(configurations["latency_ratio"].mean()), 6),
        "latency_ratio_std": round(float(configurations["latency_ratio"].std(ddof=1)), 6),
        "configurations": int(len(configurations)),
    }


def run_resource_constrained_experiment():
    configuration_errors = resource_configuration_errors()
    if configuration_errors:
        return {
            "status": "configuration_required",
            "errors": configuration_errors,
            "required_models": list(GGUF_MODEL_LOCATIONS),
            "required_datasets": list(MULTI_HORIZON_DATASETS),
        }
    try:
        prompts, warmups, missing_datasets = build_resource_prompts()
    except Exception as exc:
        return {"status": "failed", "errors": [resource_safe_error(exc)]}
    if missing_datasets:
        return {"status": "configuration_required", "missing_datasets": missing_datasets}
    timing_rows = []
    memory_rows = []
    errors = []
    model_names = list(GGUF_MODEL_LOCATIONS)
    for model_index, model_name in enumerate(model_names):
        model_timing_rows = []
        model_memory_rows = []
        try:
            with ResourceServer(model_name) as server:
                prepared, prepared_warmups = prepare_resource_tokens(server, prompts, warmups)
                warmup_order = ("raw", "full_stm") if model_index % 2 == 0 else ("full_stm", "raw")
                for variant in warmup_order:
                    server.complete(prepared_warmups[(MULTI_HORIZON_DATASETS[0], variant)])
                for repeat in range(RESOURCE_REPEATS):
                    pairs = [(dataset, window_id) for dataset in MULTI_HORIZON_DATASETS for window_id in range(RESOURCE_WINDOWS)]
                    random.Random(SEED + model_index * 1000 + repeat).shuffle(pairs)
                    for dataset, window_id in pairs:
                        parity = model_index + MULTI_HORIZON_DATASETS.index(dataset) + window_id + repeat
                        order = ("raw", "full_stm") if parity % 2 == 0 else ("full_stm", "raw")
                        for variant in order:
                            measurement = server.complete(prepared[(dataset, window_id, variant)])
                            model_timing_rows.append({
                                "backbone": model_name,
                                "dataset": dataset,
                                "window_id": int(window_id),
                                "repeat": int(repeat),
                                "variant": variant,
                                "latency_ms": measurement["latency_ms"],
                            })
            memory_order = ("raw", "full_stm") if model_index % 2 == 0 else ("full_stm", "raw")
            for variant_index, variant in enumerate(memory_order):
                with ResourceServer(model_name) as server:
                    prepared, prepared_warmups = prepare_resource_tokens(server, prompts, warmups)
                    server.complete(prepared_warmups[(MULTI_HORIZON_DATASETS[0], variant)])
                    pairs = [(dataset, window_id) for dataset in MULTI_HORIZON_DATASETS for window_id in range(RESOURCE_WINDOWS)]
                    random.Random(SEED + model_index * 100 + (0 if variant == "raw" else 1)).shuffle(pairs)
                    peak = 0
                    for dataset, window_id in pairs:
                        measurement = server.complete(prepared[(dataset, window_id, variant)], monitor_memory=True)
                        peak = max(peak, measurement["peak_rss_bytes"])
                    if peak > RESOURCE_MEMORY_LIMIT_BYTES:
                        raise RuntimeError(f"{model_name} exceeded the memory limit")
                    model_memory_rows.append({
                        "backbone": model_name,
                        "variant": variant,
                        "peak_rss_mb": round(float(peak / 1e6), 6),
                        "measured_requests": int(len(pairs)),
                    })
            summarize_resource_timing(model_timing_rows)
            if len(model_memory_rows) != 2 or {row["variant"] for row in model_memory_rows} != {"raw", "full_stm"}:
                raise RuntimeError(f"{model_name} does not contain both fresh-process memory phases")
            if any(row["measured_requests"] != len(MULTI_HORIZON_DATASETS) * RESOURCE_WINDOWS for row in model_memory_rows):
                raise RuntimeError(f"{model_name} memory phases must each contain 48 monitored requests")
            if any(not np.isfinite(float(row["peak_rss_mb"])) or float(row["peak_rss_mb"]) <= 0 for row in model_memory_rows):
                raise RuntimeError(f"{model_name} produced an invalid peak RSS measurement")
            timing_rows.extend(model_timing_rows)
            memory_rows.extend(model_memory_rows)
        except Exception as exc:
            errors.append({"backbone": model_name, "error": resource_safe_error(exc)})
    completed_backbones = {row["backbone"] for row in timing_rows}
    if errors or completed_backbones != set(model_names):
        return {
            "status": "failed",
            "completed_backbones": sorted(completed_backbones),
            "errors": errors,
        }
    try:
        backbone_rows, overall = summarize_resource_timing(timing_rows)
    except Exception as exc:
        return {"status": "failed", "errors": errors + [{"backbone": "summary", "error": resource_safe_error(exc)}]}
    memory_frame = pd.DataFrame(memory_rows)
    memory_summary = []
    if not memory_frame.empty:
        for backbone, group in memory_frame.groupby("backbone", sort=False):
            values = {row.variant: row.peak_rss_mb for row in group.itertuples(index=False)}
            memory_summary.append({
                "backbone": backbone,
                "raw_peak_rss_mb": values.get("raw"),
                "full_stm_peak_rss_mb": values.get("full_stm"),
            })
    return {
        "status": "ok" if not errors and len(backbone_rows) == len(model_names) and len(memory_summary) == len(model_names) else "partial",
        "protocol": {
            "llama_cpp_build": RESOURCE_LLAMA_CPP_BUILD,
            "llama_cpp_commit": RESOURCE_LLAMA_CPP_COMMIT,
            "cpu_threads": RESOURCE_CPU_THREADS,
            "gpu_acceleration": False,
            "quantization": "Q4_K_M",
            "context_tokens": RESOURCE_CONTEXT_SIZE,
            "memory_limit_gib": RESOURCE_MEMORY_LIMIT_BYTES / 1024**3,
            "input_length": RESOURCE_INPUT_LENGTH,
            "horizon": 1,
            "windows_per_dataset": RESOURCE_WINDOWS,
            "technical_repeats": RESOURCE_REPEATS,
            "latency_measure": "llama.cpp prompt_ms + predicted_ms",
            "rss_sampling_seconds": 0.005,
            "rss_unit": "decimal MB",
        },
        "latency": backbone_rows,
        "overall": overall,
        "memory": memory_summary,
        "errors": errors,
    }


print("result:", run_resource_constrained_experiment())
